> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


## Production imports (active)

All chunking functions are imported from `src/mrta/`:

```python
from mrta.core.schemas import Chunk
from mrta.ingestion.chunker import chunk_pdf, fixed_chunks, recursive_chunks, token_chunks, semantic_chunks
```

See [`../../production-ready.md`](../../production-ready.md) — Phase 02.

# Phase 2 — Chunking Strategies

**Goal:** Turn page-level text into retrievable chunks that (a) fit in an LLM context window, (b) carry enough surrounding context to be self-contained, and (c) preserve `{doc_id, page, section, chunk_id}` for citations.

We compare three strategies:

1. **Fixed-size character chunking** — naive baseline.
2. **Recursive character splitting** — LangChain's default. Splits on paragraph/sentence/word boundaries.
3. **Token-aware splitting** — uses `tiktoken` so chunks fit a known token budget.

We will see why **chunk size 500–800 tokens with ~100-token overlap** is the canonical sweet spot for research papers.


## 2.1 — Load the parsed PDF from Phase 1


In [ ]:
import os, sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

from mrta.core.schemas import PageRecord, PdfDocument
from mrta.ingestion import load_pdf

pdf = load_pdf("data/sample/attention_is_all_you_need.pdf")
print(f"Loaded {pdf.n_pages} pages from {pdf.source}")

## 2.2 — Define the chunk schema

A chunk is the smallest indexable unit in our system. Every chunk knows where it came from.


In [ ]:
from mrta.core.schemas import Chunk

## 2.3 — Strategy A: Fixed-size character chunking

The naive baseline. Just cut every N characters with M overlap. Cheap, but it splits sentences mid-word and ignores paragraph structure.


In [ ]:
from mrta.ingestion.chunker import fixed_chunks

fixed = fixed_chunks(pdf)
print(f"Fixed-size: {len(fixed)} chunks")
print("Example chunk preview:\n")
print(fixed[3].text[:300], "\n...")

## 2.4 — Strategy B: Recursive character splitting

LangChain's `RecursiveCharacterTextSplitter` tries a sequence of separators (`\n\n`, `\n`, `. `, ` `, `""`) and falls back to finer ones only if the chunk is still too large. Result: clean paragraph/sentence boundaries.


In [ ]:
from mrta.ingestion.chunker import recursive_chunks

recursive = recursive_chunks(pdf)
print(f"Recursive: {len(recursive)} chunks")
print("\nExample:\n", recursive[5].text[:300])

## 2.5 — Strategy C: Token-aware splitting

`tiktoken` lets us count tokens the same way the LLM will. This matters because context windows are measured in tokens, not characters, and the character-to-token ratio varies between models (English prose is ~4 chars/token).


In [ ]:
from mrta.ingestion.chunker import token_chunks

tokenized = token_chunks(pdf)
print(f"Token-aware: {len(tokenized)} chunks")
sizes = [c.n_tokens for c in tokenized]
print(f"Token sizes: min={min(sizes)}, median={sorted(sizes)[len(sizes)//2]}, max={max(sizes)}")

## 2.6 — Side-by-side comparison


In [6]:
import pandas as pd

def summarize(name: str, chunks: list[Chunk]) -> dict:
    char_lens = [len(c.text) for c in chunks]
    return {
        "strategy": name,
        "n_chunks": len(chunks),
        "median_chars": int(pd.Series(char_lens).median()),
        "p95_chars": int(pd.Series(char_lens).quantile(0.95)),
        "first_chunk_starts_clean": chunks[0].text[:1].isupper(),
    }

pd.DataFrame([
    summarize("fixed-1500/200",      fixed),
    summarize("recursive-800/100",   recursive),
    summarize("token-700/100",       tokenized),
])


,strategy,n_chunks,median_chars,p95_chars,first_chunk_starts_clean
0,fixed-1500/200,38,1500,1500,True
1,recursive-800/100,61,745,793,True
2,token-700/100,25,1411,3051,True


## 2.7 — Why these defaults?

| Parameter      | Default       | Reasoning                                                                                          |
|----------------|---------------|----------------------------------------------------------------------------------------------------|
| `chunk_size`   | 500–800 tokens| Large enough to contain a complete idea (a paragraph or two), small enough to leave room for ~3–5 retrieved chunks + the question + the answer in a 4k–8k context window. |
| `overlap`      | ~15% of size  | Lets information that straddles a chunk boundary appear in both, so retrieval is more robust.       |
| Separator order| `\n\n` → `\n` → `. ` → ` ` | Paragraph > line > sentence > word. Aim to break at semantic boundaries first.    |

If you swap to a model with a long context (e.g. 128k), you can go up to 1500–2000 token chunks and retrieve fewer of them. But beware: longer chunks dilute embedding-based retrieval ("lost in the middle" effect).


## 2.8 — Semantic chunking (advanced)

The strategies above are *structural*. A more advanced approach is **semantic chunking**: greedily merge adjacent sentences as long as their embeddings stay similar, then start a new chunk when similarity drops. This produces topic-coherent chunks.

We sketch it here using sentence embeddings; we will not make it the default because it is slower and benefits saturate around 700-token chunks for short papers.


In [ ]:
from mrta.ingestion.chunker import semantic_chunks

# semantic_chunks is slower — run only the first 5 pages for the demo.
import copy
demo_pdf = pdf.model_copy(deep=True)
demo_pdf.pages = pdf.pages[:5]
semantic = semantic_chunks(demo_pdf)
print(f"Semantic: {len(semantic)} chunks over first 5 pages")
print("\nExample:\n", semantic[2].text[:300])

## 2.9 — Save the canonical chunks for downstream notebooks

We will use the **token-aware** chunks (`token_chunks`, size=700, overlap=100) as the default for Phase 3 onwards. Saving them to `data/processed/` so we do not have to recompute.


In [8]:
import json
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

chunks_path = processed_dir / "chunks.jsonl"
with chunks_path.open("w", encoding="utf-8") as f:
    for c in tokenized:
        f.write(c.model_dump_json() + "\n")

print(f"Saved {len(tokenized)} chunks to {chunks_path}")
print(f"File size: {chunks_path.stat().st_size/1024:.1f} KB")


Saved 25 chunks to data/processed/chunks.jsonl
File size: 48.4 KB


## What you learned

- Three chunking strategies and their tradeoffs.
- Why token-aware splitting is the production default.
- The `Chunk` schema with `chunk_id`, `page`, `doc_id` for citations.
- A semantic chunking sketch for advanced use.
- How to persist chunks to JSONL.

## Exercises

1. Implement section-aware chunking: parse Markdown-like headings (`Abstract`, `Introduction`, `Methods`) and never let a chunk straddle a section boundary.
2. Plot a histogram of chunk character lengths for each strategy.
3. Tune `chunk_size` to 400, 800, 1200 — which is best? You will revisit this in Phase 9 with real eval metrics.

## Next: [Phase 3 — Embeddings & FAISS](./2026-05-25-phase03-embeddings-and-faiss.ipynb)
